# Final Assignment: Time Series and Sequential Data
**Author**: Avik Chatterjee
**Date**: April 2026

Ensure you have run `prep_data.py` or placed `stock_prices.csv` and `chat_logs.csv` into the `data/` directory.

### AI Usage Policy & Critique
**Prompt:** "Write a PyTorch LSTM class for predicting the next time step in a sequence, taking input shape (batch_size, sequence_length, hidden_dim). Write a manual RNN BPTT verifying PyTorch autograd."
**Critique:** The AI returned useful code for verifying the RNN gradients. I had to fix the manual chain rule calculation for `dW_hh` because the AI incorrectly scaled the gradient loss term across all time steps, neglecting the sequential dependency and activation derivative (`1 - tanh^2`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import random
import warnings

warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


## 🟢 Easy — Required · Sub-steps 1 & 2

### Sub-step 1: Stock Price Dataset
**Window Size Justification**: A window size of 21 days is chosen. 21 trading days represents approximately one month of trading activity, which captures typical monthly cycles and provides enough historical context for the model.

**Split Strategy Justification**: The only acceptable split for time-series is a chronological holdout (e.g., first 80% train, last 20% test). If we used a random split, data leakage would occur. Future data points correlate with past points and share macro-economic conditions. Randomly sampling train/test would leak future price trends into the training set, causing the model to achieve falsely optimistic performance during evaluation.


In [ ]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_sequences(data: np.ndarray, window_size: int):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

def process_stock_data(filepath: str, window_size: int = 21, train_ratio: float = 0.8):
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"Skipping execution: {filepath} not found.")
        return None, None, None, None, None
        
    prices = df['Close'].values.reshape(-1, 1)
    
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(prices)
    
    X, y = create_sequences(scaled_data, window_size)
    
    split_idx = int(len(X) * train_ratio)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    return X_train, X_test, y_train, y_test, scaler

WINDOW_SIZE = 21
BATCH_SIZE = 32

X_train, X_test, y_train, y_test, stock_scaler = process_stock_data("data/stock_prices.csv", WINDOW_SIZE)
if X_train is not None:
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


### Sub-step 2: Chat Logs Dataset
The `timestamp` column failed to parse initially because it possessed non-homogeneous datetime formats (mixing ISO `YYYY-MM-DD` and literal date strings `MM/DD/YYYY`). We utilize Pandas `format='mixed'` to flexibly infer dates per row.

**EDA & Churn Signal**:
Grouping interactions by `customer_id` reveals that customers with 'cancel' intents or a high frequency of messages over a short duration exhibit a vastly higher churn risk compared to normal-volume users.


In [ ]:
def process_chat_data(filepath: str):
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"Skipping execution: {filepath} not found.")
        return None, None
        
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', errors='coerce')
    
    df['has_cancel'] = df['intent'].apply(lambda x: 1 if 'cancel' in str(x).lower() else 0)
    
    agg = df.groupby('customer_id').agg(
        num_interactions=('timestamp', 'count'),
        cancel_events=('has_cancel', 'sum'),
        churn=('churn', 'max')
    ).reset_index()
    
    return df, agg

df_chat, df_chat_agg = process_chat_data("data/chat_logs.csv")
if df_chat_agg is not None:
    print("Chat Logs EDA snippet:")
    print(df_chat_agg.groupby('churn')[['num_interactions', 'cancel_events']].mean())


## 🟡 Medium — Required · Sub-steps 3, 4 & 5

### Sub-step 3: LSTM Stock Prediction
**Architecture Justification**: We employ a single-layer LSTM. Financial series are notoriously noisy and small in feature space; deeper LSTMs quickly overfit. 
**Trading Metric**: **Mean Absolute Percentage Error (MAPE)**. In quantitative finance, proportional errors matter more than relative mean squared errors. A viable model must consistently maintain low MAPE and beat a random-walk to cover transaction costs.


In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_size: int = 1, hidden_size: int = 32, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :]) 
        return out

def train_lstm_model(X_tr, y_tr, epochs: int = 5, lr: float = 0.01):
    train_loader = DataLoader(StockDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    model = StockLSTM()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
    return model

if X_train is not None:
    lstm_model = train_lstm_model(X_train, y_train)
    lstm_model.eval()

    with torch.no_grad():
        y_pred_scaled = lstm_model(torch.tensor(X_test, dtype=torch.float32)).numpy()
        
    y_test_inv = stock_scaler.inverse_transform(y_test)
    y_pred_inv = stock_scaler.inverse_transform(y_pred_scaled)

    def calculate_mape(y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    lstm_mape = calculate_mape(y_test_inv, y_pred_inv)
    print(f"LSTM Test MAPE: {lstm_mape:.2f}%")


### Sub-step 4: Churn Prediction Strategy
For churn prediction, a **tabular Random Forest model is employed**. Since chat interactions often involve isolated requests rather than heavily context-dependent linguistic loops, calculating engineered aggregations (like cancel_events or total message frequency) encapsulates the most determinative signals for churn. Sequence models (like LSTMs) risk overfitting here without massive volumes of sequential dependency data.

**Metric Choice**: `F1-Score` and `ROC-AUC`.


In [ ]:
from sklearn.model_selection import train_test_split

def train_churn_model(df_agg):
    X = df_agg[['num_interactions', 'cancel_events']]
    y = df_agg['churn']
    
    X_ctr, X_cte, y_ctr, y_cte = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf.fit(X_ctr, y_ctr)
    
    preds = rf.predict(X_cte)
    probs = rf.predict_proba(X_cte)[:, 1]
    
    f1 = f1_score(y_cte, preds)
    roc_auc = roc_auc_score(y_cte, probs)
    print(f"Tabular Model F1-Score: {f1:.3f}")
    print(f"Tabular Model ROC-AUC: {roc_auc:.3f}")
    return rf, X

if df_chat_agg is not None:
    churn_model, full_features = train_churn_model(df_chat_agg)


### Sub-step 5: Business Utility and Cost Model
**Cost Model Analysis**: 
Let Contact Cost = $10 (Support effort)
Let Missed Churn Cost = $100 (Lost customer lifetime value)

Expected Profit = (Probability * Benefit of preventing churn) - Cost of Contact
At break-even, P * $100 = $10 -> **P = 0.10**.
Customers with a churn risk >= 10% are highly cost-effective to contact.


In [ ]:
def generate_risk_list(model, df_agg, features, threshold: float = 0.1):
    df_agg['churn_prob'] = model.predict_proba(features)[:, 1]
    outreach_list = df_agg[df_agg['churn_prob'] >= threshold].sort_values('churn_prob', ascending=False)
    
    print(f"At a threshold of {threshold}, flagged for contact: {len(outreach_list)}")
    return outreach_list

if df_chat_agg is not None:
    risk_ranked_df = generate_risk_list(churn_model, df_chat_agg, full_features, threshold=0.10)
    print(risk_ranked_df.head(3))


## 🔴 Hard — Optional · Sub-steps 6 & 7

### Sub-step 6: Autoregressive Baseline vs LSTM
The baseline linearly maps past `k` observations. LSTMs often map tomorrow's price as today's price due to random-walk characteristics. A naïve autoregressive linear regression frequently matches the LSTM without over-parameterization.


In [ ]:
from sklearn.linear_model import LinearRegression

def compare_models(X_tr, y_tr, X_te, y_te, y_te_inv, scaler):
    X_tr_flat = X_tr.reshape((X_tr.shape[0], -1))
    X_te_flat = X_te.reshape((X_te.shape[0], -1))
    
    ar_model = LinearRegression()
    ar_model.fit(X_tr_flat, y_tr)
    
    ar_preds = ar_model.predict(X_te_flat)
    ar_preds_inv = scaler.inverse_transform(ar_preds)
    
    ar_mape = calculate_mape(y_te_inv, ar_preds_inv)
    print(f"Autoregressive Baseline Test MAPE: {ar_mape:.2f}%")

if X_train is not None:
    compare_models(X_train, y_train, X_test, y_test, y_test_inv, stock_scaler)


### Sub-step 7: Manual BPTT and Vanishing Gradients (A PyTorch Verification)
Below is a purely explicit chain-rule derivation of backpropagation through time compared against PyTorch Autograd. It shows that as sequence lengths enlarge, gradient magnitudes shrink exponentially.


In [ ]:
def manual_bptt_test(seq_length=5):
    # Setup dimensions
    hidden_size = 1
    input_size = 1
    
    # 1. Autograd Reference
    W_xh = torch.tensor([[0.5]], requires_grad=True)
    W_hh = torch.tensor([[0.8]], requires_grad=True) # Ensure spectral radius < 1
    
    x = torch.ones((seq_length, input_size))
    h = torch.zeros((1, hidden_size))
    
    hidden_states = []
    
    # Forward Pass
    for t in range(seq_length):
        h = torch.tanh(x[t] @ W_xh + h @ W_hh)
        hidden_states.append(h)
    
    loss = hidden_states[-1].sum() # Output is the final sequence hidden state
    loss.backward()
    
    auto_W_hh_grad = W_hh.grad.item()
    auto_W_xh_grad = W_xh.grad.item()
    
    # 2. Manual BPTT Setup
    W_hh_np = 0.8
    W_xh_np = 0.5
    h_np = np.zeros(1)
    
    h_states = [h_np]
    for t in range(seq_length):
        z = x[t].item() * W_xh_np + h_np * W_hh_np
        h_np = np.tanh(z)
        h_states.append(h_np)
        
    # Backward pass manually
    dW_hh = 0.0
    dW_xh = 0.0
    dh_next = 1.0 # dL/dh_T = 1
    
    for t in reversed(range(1, seq_length + 1)):
        # Derivative of tanh: 1 - tanh(z)^2 = 1 - h_t^2
        dtanh = 1.0 - h_states[t]**2 
        
        # Local gradients
        dz = dh_next * dtanh
        dW_hh += dz * h_states[t-1]
        dW_xh += dz * 1.0 # x[t] is 1.0
        
        dh_next = dz * W_hh_np
        
    print(f"Seq Length: {seq_length}")
    print(f"Autograd dW_hh: {auto_W_hh_grad:.4f} | Manual dW_hh: {dW_hh[0]:.4f}")
    print(f"Autograd dW_xh: {auto_W_xh_grad:.4f} | Manual dW_xh: {dW_xh[0]:.4f}")
    
    return float(dW_xh[0])

# Simulate Vanishing Gradients
seq_lengths = [5, 10, 20, 30, 50]
grads = []
for l in seq_lengths:
    grads.append(manual_bptt_test(l))

plt.figure(figsize=(6,4))
plt.plot(seq_lengths, grads, marker='o')
plt.title("Vanishing Gradient Magnitude vs Sequence Length")
plt.xlabel("Sequence Length")
plt.ylabel("Gradient (dW_xh)")
plt.grid(True)
plt.show()
